# Notebook de Tratamento (13cat)

Parte dos 7 CSVs do notebook 1 e aplica os tratamentos, na versão que **mantém as 13 categorias originais de `losses`**.

**Pré-requisito:** rodar o notebook 1 antes.

> Existem dois notebooks de tratamento idênticos, que diferem **apenas** no tratamento de `losses`: `tratamento_datasets_13cat` e `tratamento_datasets_perfil`. Rode um ou outro (ou ambos, para comparar) — cada um gera seu próprio conjunto de arquivos de saída.

### Tratamentos aplicados
1. Excluir `Oven RTD Connector Body` do Mainback (sensor com defeito).
2. Excluir `AC Blower Inlet Max` e `AC Blower Inlet Avg` do Main Vent Sys (92,9% de nulos).
3. Remover componentes sem nenhuma medição (colunas 100% nulas) e ajustar a tabela de limites.
4. **One-Hot Encoding** das features (valores 0/1), removendo `run_idx`.

Os IGBTs (1–16) e Reso Caps (1–8) **não são agregados** — a média mascararia estouros individuais de limite.

### Tratamento de `losses` nesta versão
A coluna `losses` é usada **como está**, com suas 13 categorias originais (após padronização do notebook 1). Um ensaio sem registro de losses (`NaN`) fica com todas as colunas de losses zeradas — situação que ocorre em 6 ensaios do modo Broil.

Esta é a abordagem que segue a orientação original do professor. A versão alternativa (`tratamento_datasets_perfil`) agrupa as categorias para comparação.


## 1. Configuração e leitura

Carregamos os 7 datasets gerados pelo notebook 1. O filtro no `glob` ignora arquivos com sufixo `_13cat` ou `_perfil` — assim, se este notebook já tiver rodado antes e deixado suas saídas na pasta, ele não tenta reprocessá-las por engano (foi um erro que já aconteceu).

Cada dataset vira uma entrada no dicionário `datasets`, identificada pelo nome da área.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import pandas as pd
import glob

FEATURES = ["blower_direction", "housing", "board_position", "run_case",
            "losses", "deflector_hole", "material_coil_support"]

# Familia do motivo declarado, montada na preparacao. Os seis indicadores de
# palavra anteriores davam lugar a sobreposicao entre colunas; estes seis niveis
# sao mutuamente exclusivos. Ja vem em escala binaria e nao passam pelo One-Hot.
#
# Medicao que sustenta a troca, sob o protocolo do projeto e com floresta e
# gradiente: o erro cai de 6,670 para 6,146 graus, ganho de 0,524 com intervalo
# de 0,384 a 0,670 e positivo nas tres sementes, e as aprovacoes indevidas caem
# de 35 para 34 com a deteccao subindo de 81,9 para 82,4 por cento.
#
# As quatro colunas de ordem de chaveamento permanecem, mas foram medidas como
# prejudiciais: com elas a base sobe de 6,646 para 6,705 graus, e a auditoria por
# atribuicao lhes da 1,2 por cento em media, nula em cinco das sete regioes.
EXTRAS = ["reason_air_path", "reason_heatsink_cover", "reason_plate",
          "reason_position_study", "reason_other", "reason_not_declared",
          # Ordem de chaveamento dos IGBTs, extraida do rotulo de perda na preparacao.
          "loss_profile_order_front", "loss_profile_order_back",
          "loss_profile_order_not_declared", "loss_profile_order_not_applicable"]

arquivos = sorted([f for f in glob.glob("dataset_*.csv")
                   if "_13cat" not in f and "_perfil" not in f])
print("Datasets encontrados:", len(arquivos))

datasets = {}
for arq in arquivos:
    nome = arq.replace("dataset_", "").replace(".csv", "")
    datasets[nome] = pd.read_csv(arq)
    print(f"  {nome:16s} -> {datasets[nome].shape}")


## 2. Excluir `Oven RTD Connector Body` (Mainback)

Esse sensor tem todas as leituras entre 198 °C e 273 °C, sempre acima do limite de segurança de 150 °C. O padrão é consistente com **defeito de calibração** — o sensor mede errado, não é o componente que está de fato tão quente. Mantê-lo ensinaria o modelo a prever temperaturas fisicamente inválidas. A exclusão afeta só o Mainback.

In [ ]:
if "Oven RTD Connector Body" in datasets["mainback"].columns:
    print(datasets["mainback"]["Oven RTD Connector Body"].describe().round(2))
    datasets["mainback"] = datasets["mainback"].drop(columns=["Oven RTD Connector Body"])
    print("\nExcluído. Novo shape:", datasets["mainback"].shape)


## 3. Excluir colunas com excesso de nulos (Main Vent Sys)

`AC Blower Inlet Max` e `AC Blower Inlet Avg` têm 52 de 56 valores nulos (92,9%). Com apenas 4 leituras cada, não há informação suficiente para o modelo aprender qualquer padrão, e a alta proporção de vazios pode prejudicar a predição. Por isso as duas colunas são removidas do Main Vent Sys.

In [ ]:
cols = [c for c in ["AC Blower Inlet Max", "AC Blower Inlet Avg"]
        if c in datasets["main_vent_sys"].columns]
if cols:
    datasets["main_vent_sys"] = datasets["main_vent_sys"].drop(columns=cols)
    print("Excluídas:", cols, "->", datasets["main_vent_sys"].shape)


## 4. Remover componentes sem nenhuma medição

Alguns componentes não têm leitura de temperatura em nenhum dos 56 ensaios — aparecem como coluna inteiramente nula (o notebook 1 os preserva de propósito, para que essa ausência fique visível aqui em vez de sumir em silêncio).

Um componente sem nenhuma medição não pode ser aprendido nem avaliado, então ele é removido tanto dos datasets quanto da tabela de limites (onde a linha ficaria órfã na comparação pós-predição). A varredura é dinâmica: pega qualquer coluna 100% nula, sem precisar listar nomes.

In [ ]:
limites = pd.read_csv("limites_ativos.csv")
removidos = []
for nome, df in datasets.items():
    alvos = [c for c in df.columns if c not in FEATURES + EXTRAS + ["run_idx"]]
    vazios = [c for c in alvos if df[c].isna().all()]
    if vazios:
        datasets[nome] = df.drop(columns=vazios)
        removidos += vazios
        print(f"  {nome}: removidos {vazios}")
if removidos:
    limites = limites[~limites["tc_name"].isin(removidos)]
else:
    print("Nenhum componente 100% nulo.")
print("Componentes na tabela de limites:", len(limites))


## 5. Manter as 13 categorias de `losses`

Nesta versão, a coluna `losses` segue **como está** (já padronizada pelo notebook 1), conforme a orientação original do professor. Nenhuma categoria é agrupada.

Um ensaio **sem registro** de losses (`NaN`) ficará, no one-hot da próxima seção, com todas as colunas de losses em 0. Isso é diferente de `no losses` (perda zero, caso do Pyro), que tem sua própria coluna — os dois não se confundem.

A célula abaixo só imprime as categorias presentes, para conferência.

In [ ]:
# Conferência: categorias presentes
ex = datasets["coil_temps"]["losses"]
print("Categorias de losses (coil_temps):", ex.nunique(dropna=True), "categorias")
print(ex.value_counts(dropna=False).to_string())


## 6. One-Hot Encoding

Cada coluna de texto (feature de design) é transformada em várias colunas de **0 e 1**, uma por categoria — porque o modelo só faz conta com número, não entende a palavra "pyro" ou "simone".

Duas colunas ficam de fora: `run_idx` (é só um identificador, não uma característica do design) e as colunas de componentes (são o alvo da predição). O `run_idx` é **removido agora**, antes do treino, para não haver risco de entrar como feature por engano.

O `.astype(int)` garante que o resultado seja 0/1 e não `True`/`False` (versões recentes do pandas geram booleano por padrão). Ensaios com `losses` ausente ficam com todas as colunas de losses em 0.

In [ ]:
SUFIXO = "_13cat"

for nome, df in datasets.items():
    alvos = [c for c in df.columns if c not in FEATURES + EXTRAS + ["run_idx"]]
    dummies = pd.get_dummies(df[FEATURES], columns=FEATURES, dummy_na=False).astype(int)
    presentes = [c for c in EXTRAS if c in df.columns]
    out = pd.concat([dummies, df[presentes], df[alvos]], axis=1)   # run_idx sai aqui
    arq = f"dataset_{nome}{SUFIXO}.csv"
    out.to_csv(arq, index=False)
    n_losses = len([c for c in out.columns if c.startswith("losses_")])
    print(f"  {arq:34s} -> {out.shape}  ({n_losses} colunas de losses)")

limites.to_csv("limites_ativos.csv", index=False)


## 7. Conferência final

Duas verificações rápidas de sanidade: que cada ensaio tem no máximo uma categoria de losses marcada (soma 0 quando não há registro, 1 quando há), e que as colunas do one-hot estão como números inteiros (0/1), não booleanos.

In [ ]:
# Um ensaio por vez: soma das colunas de losses deve ser 0 (sem registro) ou 1 (com registro)
df_check = pd.read_csv(f"dataset_coil_temps{SUFIXO}.csv")
losses_cols = [c for c in df_check.columns if c.startswith("losses_")]
soma = df_check[losses_cols].sum(axis=1)
print("Colunas de losses:", losses_cols)
print(f"Ensaios com registro de losses (soma=1): {(soma==1).sum()}")
print(f"Ensaios sem registro (soma=0):          {(soma==0).sum()}")
print()
# Valores 0/1, não booleanos
print("Tipo das colunas one-hot:", df_check[losses_cols].dtypes.unique().tolist())


## 8. Resumo

Saídas: `dataset_<area>_13cat.csv` (7 arquivos) + `limites_ativos.csv` filtrado.

| Tratamento | Onde |
|---|---|
| Oven RTD Connector Body excluído | Mainback |
| AC Blower Inlet Max/Avg excluídos | Main Vent Sys |
| Componentes 100% nulos removidos | Todos + limites |
| IGBTs e Reso Caps mantidos individuais | IPC Comp |
| One-Hot (0/1), run_idx removido | Todos |
| **losses: 13 categorias** | Todos |

> Housing "open - updated" já foi unificado no notebook 1. Os nulos restantes estão nas colunas-alvo (temperaturas) e serão tratados na fase de modelagem com o professor.


## 9. Versão tratada: ausentes preenchidos e IGBTs em quatro grupos

Esta seção grava uma versão alternativa dos sete conjuntos, em arquivos novos, sem alterar os arquivos `dataset_<area>_13cat.csv`, que continuam sendo a entrada da modelagem e da formulação entregue.

Dois tratamentos são aplicados, nesta ordem.

1. Os dezesseis IGBTs de IPC Comp são substituídos por quatro colunas, cada uma com a média do seu grupo de quatro transistores: 1 a 4, 5 a 8, 9 a 12 e 13 a 16. Os grupos foram recuperados do próprio dado por aglomeração hierárquica, conforme a seção 8.5 do relatório da sprint. Quando um transistor do grupo não tem leitura num ensaio, a média usa os demais.
2. Os valores ausentes que restam nas colunas de temperatura são preenchidos com a mediana da coluna, calculada apenas sobre os 46 ensaios de desenvolvimento. Os dez ensaios do conjunto de teste reservado não entram no cálculo, para que nenhuma informação deles chegue ao treino.

O limite de cada grupo de IGBTs é o menor limite entre os seus transistores, e os quatro grupos são acrescentados a um arquivo de limites próprio, `limites_ativos_tratado.csv`.

Duas ressalvas acompanham esta versão. A primeira é que o agrupamento dos IGBTs foi medido no relatório da sprint (seção 10.6) e piora o erro de IPC Comp em cerca de 1,0 °C, porque reduz o volume de treino; além disso, a média do grupo pode esconder um transistor acima do limite, o que afeta a triagem. A segunda é que os valores preenchidos nos ensaios do conjunto de teste reservado não são medições e não devem ser usados como valor real na avaliação.

Saídas: `dataset_<area>_13cat_tratado.csv` (7 arquivos) e `limites_ativos_tratado.csv`.

In [ ]:
import json
import os

import numpy as np
import pandas as pd

AREAS_TRATADAS = ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling",
                  "main_vent_sys", "mainback", "oven_hmi"]
GRUPOS_IGBT = {f"IGBT {a} a {a + 3}": [f"IGBT {i}" for i in range(a, a + 4)]
               for a in (1, 5, 9, 13)}
PREFIXOS_ENTRADA = tuple(f"{f}_" for f in ["blower_direction", "housing", "board_position",
                                            "run_case", "losses", "deflector_hole",
                                            "material_coil_support"]) + ("reason_", "loss_profile_order_")


def localiza(nome):
    """Arquivo ao lado do caderno (Colab) ou na pasta de preparacao do repositorio."""
    for pasta in (".", "../colabs_S3/preparacao_testes", "notebooks/colabs_S3/preparacao_testes"):
        caminho = os.path.join(pasta, nome)
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(f"{nome} nao encontrado; envie o arquivo junto com o caderno.")


particao = json.load(open(localiza("particao_treino_teste.json"), encoding="utf-8"))
RESERVADOS = set(particao["run_idx_teste"])

limites_base = pd.read_csv("limites_ativos.csv")
novos_limites = []
resumo_tratado = []

for area in AREAS_TRATADAS:
    df = pd.read_csv(f"dataset_{area}_13cat.csv")
    assert len(df) == 56, f"{area}: esperados 56 ensaios"
    reservado = np.isin(np.arange(len(df)), list(RESERVADOS))   # a ordem das linhas e o run_idx
    entradas = [c for c in df.columns if c.startswith(PREFIXOS_ENTRADA)]
    alvos_antes = [c for c in df.columns if c not in entradas]
    nulos_antes = int(df[alvos_antes].isna().sum().sum())

    # 1. IGBTs em quatro grupos
    grupos_aplicados = 0
    for grupo, membros in GRUPOS_IGBT.items():
        presentes = [m for m in membros if m in df.columns]
        if not presentes:
            continue
        assert len(presentes) == 4, f"{area}: grupo {grupo} incompleto: {presentes}"
        df[grupo] = df[presentes].mean(axis=1, skipna=True)
        df = df.drop(columns=presentes)
        lim = limites_base.loc[limites_base["tc_name"].isin(presentes)]
        novos_limites.append({"tc_name": grupo, "key_area": lim["key_area"].iloc[0],
                              "active_limit": lim["limite_num"].min(),
                              "limite_num": lim["limite_num"].min(),
                              "apenas_monitorar": bool(lim["apenas_monitorar"].all())})
        grupos_aplicados += 1

    # 2. Mediana dos ensaios de desenvolvimento nos ausentes restantes
    alvos = [c for c in df.columns if c not in entradas]
    preench_dev = int(df.loc[~reservado, alvos].isna().sum().sum())
    preench_res = int(df.loc[reservado, alvos].isna().sum().sum())
    medianas = df.loc[~reservado, alvos].median()
    assert medianas.notna().all(), f"{area}: coluna sem leitura nos ensaios de desenvolvimento"
    df[alvos] = df[alvos].fillna(medianas)

    assert df.isna().sum().sum() == 0, f"{area}: restaram nulos"
    df.to_csv(f"dataset_{area}_13cat_tratado.csv", index=False)
    resumo_tratado.append({"area": area, "alvos antes": len(alvos_antes), "alvos depois": len(alvos),
                           "grupos de IGBT": grupos_aplicados, "nulos antes": nulos_antes,
                           "preenchidos no desenvolvimento": preench_dev,
                           "preenchidos no reservado": preench_res,
                           "arquivo": f"dataset_{area}_13cat_tratado.csv"})

limites_tratado = limites_base[~limites_base["tc_name"].isin(sum(GRUPOS_IGBT.values(), []))]
limites_tratado = pd.concat([limites_tratado, pd.DataFrame(novos_limites)], ignore_index=True)
limites_tratado.to_csv("limites_ativos_tratado.csv", index=False)

print("Gravado: limites_ativos_tratado.csv com", len(novos_limites), "grupos de IGBT")
pd.DataFrame(resumo_tratado).set_index("area")

## 10. Versão perfil e partição, gravadas nesta pasta

Os cadernos desta pasta leem apenas arquivos CSV do próprio colabs_S4. Duas entradas ainda vinham de outras pastas, e passam a ser geradas aqui.

A primeira é a versão da base em que a coluna de perdas é reduzida ao perfil de perda, com três níveis: simone, miro e no losses. Ela era lida do colabs_S3, onde foi gerada na terceira sprint, e aquela versão tem duas diferenças em relação à base atual: codifica o motivo do ensaio pelos seis indicadores antigos, que se sobrepõem, e mantém o sensor Thermistor do Oven HMI, que é cópia exata do Microchip e contava as mesmas 45 leituras duas vezes. A versão gravada aqui parte dos arquivos 13cat desta pasta, de modo que herda o motivo em seis níveis exclusivos e o descarte do sensor duplicado; muda apenas a codificação das perdas.

A segunda é o arquivo da partição em CSV, usado pelos cadernos de busca de hiperparâmetros. Ele é escrito a partir do arquivo JSON da partição, que é a fonte registrada no repositório, e contém as mesmas colunas de ensaio, conjunto, dobra e tipo de ensaio.

Saídas: `dataset_<area>_perfil.csv` (7 arquivos) e `particao_treino_teste.csv`.

In [ ]:
import json
import os

import numpy as np
import pandas as pd


def localiza_json(nome):
    """A particao JSON fica ao lado do caderno (Colab) ou na pasta de preparacao do repositorio."""
    for pasta in (".", "../colabs_S3/preparacao_testes", "notebooks/colabs_S3/preparacao_testes"):
        caminho = os.path.join(pasta, nome)
        if os.path.exists(caminho):
            return caminho
    raise FileNotFoundError(f"{nome} nao encontrado; envie o arquivo junto com o caderno.")


def perfil_de_perda(categoria):
    """Mesma regra do tratamento perfil da terceira sprint."""
    if categoria == "no losses":
        return "no losses"
    if "simone" in categoria and "miro" not in categoria:
        return "simone"
    if "miro" in categoria:
        return "miro"
    return categoria


resumo_perfil = []
for area in ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling", "main_vent_sys", "mainback", "oven_hmi"]:
    df = pd.read_csv(f"dataset_{area}_13cat.csv")
    perdas = [c for c in df.columns if c.startswith("losses_")]
    niveis = {}
    for c in perdas:
        niveis.setdefault(perfil_de_perda(c[len("losses_"):]), []).append(c)
    agrupadas = pd.DataFrame({f"losses_{n}": (df[cols].sum(axis=1) > 0).astype(int)
                              for n, cols in sorted(niveis.items())})
    # a coluna das perdas fica no mesmo lugar em que estavam as treze
    pos = df.columns.get_loc(perdas[0])
    out = pd.concat([df.iloc[:, :pos], agrupadas, df.drop(columns=perdas).iloc[:, pos:]], axis=1)
    assert out.shape[0] == 56 and (agrupadas.sum(axis=1) <= 1).all(), f"{area}: perfil inconsistente"
    assert set(df.columns) - set(perdas) == set(out.columns) - set(agrupadas.columns)
    out.to_csv(f"dataset_{area}_perfil.csv", index=False)
    resumo_perfil.append({"area": area, "colunas de perdas": f"{len(perdas)} -> {agrupadas.shape[1]}",
                          "forma": out.shape, "arquivo": f"dataset_{area}_perfil.csv"})

P = json.load(open(localiza_json("particao_treino_teste.json"), encoding="utf-8"))
teste = set(P["run_idx_teste"])
particao_csv = pd.DataFrame({
    "run_idx": range(P["n_ensaios"]),
    "conjunto": ["teste" if r in teste else "treino" for r in range(P["n_ensaios"])],
    "dobra": [P["dobra_por_run_idx"][str(r)] for r in range(P["n_ensaios"])],
    "run_case": [P["run_case_por_run_idx"][str(r)] for r in range(P["n_ensaios"])],
})
assert particao_csv["dobra"].isna().sum() == len(teste) and particao_csv["dobra"].nunique() == P["n_dobras"]
particao_csv.to_csv("particao_treino_teste.csv", index=False)
print(f"Gravado: particao_treino_teste.csv ({len(particao_csv)} ensaios, "
      f"{(particao_csv['conjunto'] == 'teste').sum()} reservados, {P['n_dobras']} dobras)")
pd.DataFrame(resumo_perfil).set_index("area")